In [11]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_295K_278464_17O_opt_rPBE_magres.magres') #latest magres file from 2025

In [12]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [13]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [14]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [15]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

14N1 sigma:
 [[186.32224076   1.75087565  -3.25352149]
 [ -0.2167895  178.5121695   -4.97904704]
 [ -3.72963411  -5.50979796 191.54487214]]

14N2 sigma:
 [[186.32224076  -1.75087565   3.25352149]
 [  0.2167895  178.5121695   -4.97904704]
 [  3.72963411  -5.50979796 191.54487214]]

14N3 sigma:
 [[186.32224076   1.75087565   3.25352149]
 [ -0.2167895  178.5121695    4.97904704]
 [  3.72963411   5.50979796 191.54487214]]

14N4 sigma:
 [[186.32224076  -1.75087565  -3.25352149]
 [  0.2167895  178.5121695    4.97904704]
 [ -3.72963411   5.50979796 191.54487214]]



In [16]:
for atom in atoms.species('N'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

14N1 sigma:
 1.3257017931787798

14N2 sigma:
 1.3257017931787878

14N3 sigma:
 1.3257017931788184

14N4 sigma:
 1.325701793178821



In [17]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

atom_label = 0                                      # atom for which parameters are wanted
CS_total[:,:] = atoms.species('N').ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species('N')[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf
Q = 0.0204 #electric quadrupole moment for O17 in barn
V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.372 -0.242 -1.07 ]
 [-0.242 -0.301  0.271]
 [-1.07   0.271 -0.071]]

CS Tensor:
 [[186.322   1.751  -3.254]
 [ -0.217 178.512  -4.979]
 [ -3.73   -5.51  191.545]]

CS isotropic Tensor:
 [[185.46   0.     0.  ]
 [  0.   185.46   0.  ]
 [  0.     0.   185.46]]

CS symmetric Tensor:
 [[186.322   0.767  -3.492]
 [  0.767 178.512  -5.244]
 [ -3.492  -5.244 191.545]]

CS antisymmetric Tensor:
 [[ 0.     0.984  0.238]
 [-0.984  0.     0.265]
 [-0.238 -0.265  0.   ]]


In [18]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 1.32310701 -0.94830554 -0.37480147] 

 Unsorted Eigenvectors:
 [[-0.7536816   0.61595035 -0.22927979]
 [ 0.21612981 -0.09717678 -0.97151664]
 [ 0.62068668  0.78176841  0.05988488]] 

Sorted Eigenvalues: 
 [-0.37480147 -0.94830554  1.32310701] 

Sorted Eigenvectors: 
 [[-0.22927979  0.61595035 -0.7536816 ]
 [-0.97151664 -0.09717678  0.21612981]
 [ 0.05988488  0.78176841  0.62068668]] 


For CS tensor
 Unsorted Eigenvalues:
 [194.86800489 184.86877424 176.64250326] 

 Unsorted Eigenvectors:
 [[ 0.38382932  0.92210844  0.0488986 ]
 [ 0.29821963 -0.17390344  0.93852152]
 [-0.87392225  0.34564956  0.34174008]] 

Sorted Eigenvalues: 
 [184.86877424 176.64250326 194.86800489] 

Sorted Eigenvectors: 
 [[ 0.92210844  0.0488986   0.38382932]
 [-0.17390344  0.93852152  0.29821963]
 [ 0.34564956  0.34174008 -0.87392225]] 



In [19]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.3748014685796754 -0.9483055429183216 1.3231070114979788
CSA Tensor Components δyy, δxx, δzz: 
 184.86877424211409 176.64250326172723 194.8680048899769


In [20]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        1.32311
etaq            0.433453
iso_cs (ppm)  185.46
csa (ppm)       9.40824
etas            0.874368


In [21]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.61595035 -0.22927979 -0.7536816 ]
 [-0.09717678 -0.97151664  0.21612981]
 [ 0.78176841  0.05988488  0.62068668]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
4.380406147779185 51.63370298342612 16.00105998171095 

Direction cosine csa: 

[[ 0.0488986   0.92210844  0.38382932]
 [ 0.93852152 -0.17390344  0.29821963]
 [ 0.34174008  0.34564956 -0.87392225]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
45.32586227758028 150.91768020088534 -37.845740368219154 



In [22]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -53.87444515650017 chi: 140.1087272392345 xi: -42.11177738992167 



**Rotation of tensors Crystal--> Tenon Frame**

In [23]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[186.32224076   0.76704308  -3.4915778 ]
 [  0.76704308 178.5121695   -5.2444225 ]
 [ -3.4915778   -5.2444225  191.54487214]]
CSA Tensor in Tenon Frame: 
 [[180.38553819  -6.3736537    3.57133713]
 [ -6.3736537  190.55176556  -2.02187635]
 [  3.57133713  -2.02187635 185.44197864]]
Quad Tensor in Crystal Frame: 
 [[ 0.3720871  -0.24224981 -1.07044152]
 [-0.24224981 -0.30090439  0.2713415 ]
 [-1.07044152  0.2713415  -0.07118271]]
Quad Tensor in Tenon Frame: 
 [[-0.37922593  0.08101963 -0.05874491]
 [ 0.08101963  1.29921526  0.21469567]
 [-0.05874491  0.21469567 -0.91998933]]
